In [ ]:
import numpy as np
from spafe.features.spfeats import extract_feats
import matplotlib.pyplot as plt
import numpy as np
from scipy.fftpack import rfft
import scipy.stats
from scipy.signal import stft as scipy_stft

import librosa
from tqdm import tqdm
import pandas as pd
import sklearn
import sklearn.preprocessing

from spafe.features.mfcc import mfcc, imfcc
from spafe.features.bfcc import bfcc
from spafe.features.lfcc import lfcc
from spafe.features.lpc import lpc, lpcc
from spafe.features.msrcc import msrcc
from spafe.features.ngcc import ngcc
from spafe.features.psrcc import psrcc
from spafe.features.rplp import plp, rplp
from spafe.features.gfcc import gfcc

In [ ]:
#음성 데이터 특징들을 각각 분류해 놓은 것
SP_FEATS_NAMES = [
    'duration','spectrum', 'mean_frequency', 'peak_frequency', 'frequencies_std', 'amplitudes_cum_sum', 'mode_frequency', 'median_frequency', 'frequencies_q25', 'frequencies_q75',
    'iqr', 'freqs_skewness', 'freqs_kurtosis', 'spectral_entropy', 'spectral_flatness', 'spectral_centroid', 'spectral_bandwidth', 'spectral_spread', 'spectral_rolloff', 'energy',
    'rms', 'zcr', 'spectral_mean', 'spectral_rms', 'spectral_std', 'meanfun', 'minfun', 'maxfun', 'meandom', 'mindom', 'maxdom', 'dfrange', 'modindex'
]

SP = ['spectral_centroid', 'spectral_skewness', 'spectral_kurtosis', 'spectral_entropy', 'spectral_spread', 'spectral_flatness', 'spectral_rolloff',
      'spectral_flux', 'spectral_mean', 'spectral_rms', 'spectral_std', 'spectral_variance']


#constants.py 내용
MEDIA_INFO_FEATURES = ['bit_rate']
SPECTRUM_FEATURES = ['mfcc', 'bfcc', 'lfcc', 'lpc', 'lpcc', 'msrcc', 'ngcc', 'psrcc', 'plp', 'rplp', 'gfcc'] #음성 스펙트럼 특징 추출
DROP_FEATURES = ["label","duration", "size", "spectral_bandwidth"]
SPECTRAL_COMPLEX_VALUES = ['spectral_flatness', 'spectral_centroid', 'spectral_spread'] #복소수 값을 가지는 음성 특징
TWO_DEMENSION_FEATURES = ['spectrum', 'amplitudes_cum_sum', 'energy']

SPECTRUM_FEATURES_FUNCTIONS = [mfcc, bfcc, lfcc, lpc, lpcc, msrcc, ngcc, psrcc, plp, rplp, gfcc] #음성 스펙트럼 특징 추출할 함수


In [ ]:
import numpy as np
from scipy.fftpack import rfft
import scipy.stats
from scipy.signal import stft as scipy_stft
import scipy.signal
import librosa

# STFT 함수 정의
def stft(sig, fs, nperseg=256, noverlap=None):
    f, t, Zxx = scipy_stft(sig, fs, nperseg=nperseg, noverlap=noverlap)
    return f, t, Zxx

# RFFT 함수 정의
def rfft(sig, n=None):
    return np.fft.rfft(sig, n=n)

# Fundamental Frequencies Extractor 클래스 정의
class FundamentalFrequenciesExtractor:
    def __init__(self, fs):
        self.fs = fs

    def compute_fund_freqs(self, sig):
        fourrier_transform = np.fft.fft(sig)
        magnitude_spectrum = np.abs(fourrier_transform)
        frequencies = np.fft.fftfreq(len(magnitude_spectrum), 1/self.fs)
        fundamental_freq = frequencies[np.argmax(magnitude_spectrum)]
        return fundamental_freq

# Dominant Frequencies 함수 정의
def get_dominant_frequencies(sig, fs, lower_cutoff=50, upper_cutoff=3000):
    nyquist = fs / 2
    low = lower_cutoff / nyquist
    high = upper_cutoff / nyquist
    b, a = scipy.signal.butter(1, [low, high], btype='band')
    filtered_sig = scipy.signal.lfilter(b, a, sig)
    dominant_freqs = np.fft.fftfreq(len(filtered_sig), 1/fs)
    return dominant_freqs[np.argmax(np.abs(np.fft.fft(filtered_sig)))]

def compute_fund_freqs(sig, fs):
    """
    compute fundamental frequencies.

    Args:
        centroid (float) : spectral centroid.
        spectrum (array) : spectrum array.

    Returns:
        (float) spectral spread.
    """
    # fundamental frequencies calculations
    fund_freqs_extractor = FundamentalFrequenciesExtractor(fs)
    fundamental_freq = fund_freqs_extractor.compute_fund_freqs(sig)
    return np.array([fundamental_freq])


def extract_frequency_feats(sig, fs, nfft=512):
    feats = {}

    fourrier_transform = rfft(sig, nfft)
    magnitude_spectrum = (1/nfft) * np.abs(fourrier_transform)
    power_spectrum = (1/nfft)**2 * magnitude_spectrum**2

    frequencies = np.fft.fftfreq(nfft, 1 / fs)
    positive_freqs = frequencies[:nfft//2]
    magnitude_spectrum = magnitude_spectrum[:len(positive_freqs)]
    power_spectrum = power_spectrum[:len(positive_freqs)]

    spectrum = power_spectrum
    amplitudes = power_spectrum
    amp_cumsum = np.cumsum(amplitudes)

    feats["duration"] = len(sig) / float(fs)
    feats["spectrum"] = spectrum   #2차원

    feats["mean_frequency"] = positive_freqs.sum()
    feats["peak_frequency"] = positive_freqs[np.argmax(amplitudes)]
    feats["frequencies_std"] = positive_freqs.std()
    feats["amplitudes_cum_sum"] = amp_cumsum   #2차원
    feats["mode_frequency"] = positive_freqs[amplitudes.argmax()]
    feats["median_frequency"] = np.median(positive_freqs)
    feats["frequencies_q25"] = positive_freqs[np.searchsorted(amp_cumsum, 0.25 * amp_cumsum[-1])]
    feats["frequencies_q75"] = positive_freqs[np.searchsorted(amp_cumsum, 0.75 * amp_cumsum[-1])]
    feats["iqr"] = feats["frequencies_q75"] - feats["frequencies_q25"]

    feats["freqs_skewness"] = scipy.stats.skew(positive_freqs)
    feats["freqs_kurtosis"] = scipy.stats.kurtosis(positive_freqs)

    feats["energy"] = magnitude_spectrum     #2차원

    feats["rms"] = np.sqrt(np.mean(sig**2))

    feats["zcr"] = ((sig[:-1] * sig[1:]) < 0).sum() / len(sig)

    fund_freqs = compute_fund_freqs(sig, fs)
    feats["meanfun"] = fund_freqs.mean()
    feats["minfun"] = fund_freqs.min()
    feats["maxfun"] = fund_freqs.max()

    dom_freqs = get_dominant_frequencies(sig, fs)
    feats["meandom"] = dom_freqs
    feats["mindom"] = dom_freqs
    feats["maxdom"] = dom_freqs
    feats["dfrange"] = feats["maxdom"] - feats["mindom"]
    feats["modindex"] = 0  # Placeholder, replace with appropriate calculation if available

    return feats



# 오디오 파일에서 음성특징 추출
def extract_sp_feats(file_path: str, dtype: str = "float64") -> dict:
    y, sr = librosa.load(file_path, sr=16000)  # librosa로 오디오 파일 로드

    sp_feats = extract_feats(sig=y, fs=sr)  # extract로 음성파일의 특징을 추출해서 딕셔너리로 반환
    for sp_feat_name in sp_feats:
        sp_feat_value = sp_feats[sp_feat_name]
        if isinstance(sp_feat_value, (tuple, np.ndarray, list)):
            sp_feat_value = np.array(sp_feat_value)
            sp_feats[sp_feat_name] = sp_feat_value.mean() if sp_feat_value.size > 0 else 0
        elif sp_feat_name in SPECTRAL_COMPLEX_VALUES:
            sp_feats[sp_feat_name] = np.array(sp_feat_value).real.mean()

    sp_fre_feats = extract_frequency_feats(sig=y, fs=sr)
    for sp_feat_name, sp_feat_value in sp_fre_feats.items():
        sp_feat_value = np.array(sp_feat_value)
        if sp_feat_name not in sp_feats:
            sp_feats[sp_feat_name] = sp_feat_value.mean() if sp_feat_value.size > 0 else 0
        else:
            if sp_feat_name in SPECTRAL_COMPLEX_VALUES:
                sp_feats[sp_feat_name] = sp_feat_value.real.mean()
            else:
                sp_feats[sp_feat_name] = sp_feat_value.mean() if sp_feat_value.size > 0 else 0

    return sp_feats



def extract_spectrum_data(sample: str) -> dict:
    y, sr = librosa.load(sample, sr=16000)  # librosa로 오디오 파일 로드

    spectrum_dict = {}  # 딕셔너리 생성
    spectrum_dict["signal"] = y.mean()  # signal 부분은 음성의 y값의 평균을 넣음

    for idx in range(len(SPECTRUM_FEATURES)):
        feature = SPECTRUM_FEATURES_FUNCTIONS[idx](sig=y, fs=sr)

        if idx == 3:
            lpc_1 = np.array(feature[0])
            lpc_2 = np.array(feature[1])
            spectrum_dict[SPECTRUM_FEATURES[idx]] = (lpc_1.mean() + lpc_2.mean())/2
            continue

        # 2차원 배열의 경우 전체 평균값을 구합니다
        if isinstance(feature, (np.ndarray, list)):
            feature = np.array(feature)
            if feature.ndim == 2:  # 2차원 배열인 경우
                spectrum_dict[SPECTRUM_FEATURES[idx]] = feature.mean()
            else:
                spectrum_dict[SPECTRUM_FEATURES[idx]] = feature.mean()
        else:
            spectrum_dict[SPECTRUM_FEATURES[idx]] = feature

    return spectrum_dict


def filter_features(features: dict) -> list[float]:
    filtered_features = {}  #딕셔너리 형태
    for f in features:   # features의 key값이 f에 들어감
        if f not in DROP_FEATURES:  #DROP_FEATURES에 있는것 뺴고 filtered_features딕셔너리에 추가
            filtered_features[f] = features[f]

    return filtered_features

def get_all_features_from_sample(file_path: str) -> list[float]:
    sp_feats = extract_sp_feats(file_path)
    spectrum_data = extract_spectrum_data(file_path)
    features = sp_feats | spectrum_data  #두 개의 딕셔너리를 합침
    return filter_features(features)


In [ ]:
feats = get_all_features_from_sample("./train/RUNQPNJF.ogg")
print(feats)

In [ ]:
df = pd.read_csv('./train.csv')

In [ ]:
import os

output_csv_path = 'data_set.csv'

# 기존 파일이 있는지 확인하고, 있다면 로드
if os.path.exists(output_csv_path):
    features_df = pd.read_csv(output_csv_path)
else:
    features_df = pd.DataFrame()


for _, row in tqdm(df.iterrows(), total=df.shape[0]):
    if not features_df.empty and row['id'] in features_df['id'].values:
        continue
    
    y, sr = librosa.load(row['path'], sr=16000)
    
    feature = get_all_features_from_sample(row['path'])

    target_length = 16000 * 5
    if len(y) < target_length:
        y = np.pad(y, (0, target_length - len(y)), mode='constant')
    else:
        y = y[:target_length]
    
    new_row = {'id': row['id'], 'label': 1 if row['label'] == 'real' else 0}
    new_row.update(feature) 
    
    new_df = pd.DataFrame([new_row])
    
    features_df = pd.concat([features_df, new_df], ignore_index=True)

    new_df.to_csv(output_csv_path, mode='a', header=not os.path.exists(output_csv_path), index=False)